# 🪜 04 · 漏斗分析与转化率

> 漏斗是转化分析的骨架：**曝光 → 点击 → 加购 → 下单 → 支付**。本节计算各层转化率、流失率，并定位「最大流失环节」。

## 📋 本节要解决的问题

| # | 问题 | 工具 |
|---|---|---|
| 1 | 各层转化率怎么算？ | 相邻层人数比 |
| 2 | 整体转化率怎么算？ | 首层→末层 |
| 3 | 最大流失环节在哪？ | 流失率对比 |
| 4 | 漏斗可视化 | 条形/阶梯图 |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
np.random.seed(42)


## 1️⃣ 定义漏斗数据

In [2]:
stages = ['曝光', '点击', '加购', '下单', '支付']
users = [100000, 32000, 9000, 4200, 2600]
df_funnel = pd.DataFrame({'stage': stages, 'users': users})
print(df_funnel)
assert users[0] == 100000 and users[-1] == 2600
print('✅ 数据就绪')

  stage   users
0    曝光  100000
1    点击   32000
2    加购    9000
3    下单    4200
4    支付    2600
✅ 数据就绪


## 2️⃣ 转化率与流失率计算

In [3]:
df_funnel['conv_to_next'] = df_funnel['users'].shift(-1) / df_funnel['users']
df_funnel['loss'] = 1 - df_funnel['conv_to_next']
print(df_funnel.round(3))

overall = users[-1] / users[0]
print(f'\n整体转化率（曝光→支付）: {overall:.2%}')

# 最大流失环节
worst = df_funnel.loc[df_funnel['loss'].idxmax(), 'stage']
worst_loss = df_funnel['loss'].max()
print(f'最大流失环节: {worst}（流失率 {worst_loss:.1%}）')
assert worst == '点击' and worst_loss > 0.6
print('✅ 定位到最大流失环节：点击')

  stage   users  conv_to_next   loss
0    曝光  100000         0.320  0.680
1    点击   32000         0.281  0.719
2    加购    9000         0.467  0.533
3    下单    4200         0.619  0.381
4    支付    2600           NaN    NaN

整体转化率（曝光→支付）: 2.60%
最大流失环节: 点击（流失率 71.9%）
✅ 定位到最大流失环节：点击


## 3️⃣ 漏斗图可视化

In [4]:
plt.figure(figsize=(8, 5))
colors = ['#60a5fa', '#818cf8', '#a78bfa', '#c084fc', '#e879f9']
# 以首层为 100% 绘制相对宽度
widths = np.array(users) / users[0]
y = np.arange(len(stages))[::-1]
for wi, yi, c, u in zip(widths, y, colors, users):
    plt.barh(yi, wi, height=0.6, color=c, label=f'{stages[len(stages)-1-yi]} · {u:,.0f}')
plt.yticks(y, stages[::-1])
plt.xlabel('相对人数（首层=100%）')
plt.title('电商转化漏斗')
plt.legend(loc='lower right', fontsize=9)
plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_43624\765097005.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4️⃣ 渠道漏斗对比：找出短板渠道

两个渠道同漏斗结构，各自层人数：

In [5]:
ch_a = [50000, 18000, 5200, 2400, 1500]
ch_b = [30000, 9000, 3200, 1800, 1200]
print(f'渠道 A 整体转化率: {ch_a[-1]/ch_a[0]:.2%}')
print(f'渠道 B 整体转化率: {ch_b[-1]/ch_b[0]:.2%}')

# 每层转化对比
layers = np.arange(1, 5)
rate_a = np.array([ch_a[i]/ch_a[i-1] for i in layers])
rate_b = np.array([ch_b[i]/ch_b[i-1] for i in layers])

plt.figure(figsize=(8, 4))
x = np.arange(4)
plt.bar(x-0.2, rate_a, 0.4, label='渠道 A', color='#60a5fa')
plt.bar(x+0.2, rate_b, 0.4, label='渠道 B', color='#f59e0b')
plt.xticks(x, [f'{stages[i]}→{stages[i+1]}' for i in range(4)], rotation=20)
plt.ylabel('层转化率')
plt.title('渠道漏斗各层转化率对比')
plt.legend()
plt.show()
print('→ 渠道 B 在「加购→下单」层明显偏低，优先排查')

渠道 A 整体转化率: 3.00%
渠道 B 整体转化率: 4.00%
→ 渠道 B 在「加购→下单」层明显偏低，优先排查


C:\Users\24260\AppData\Local\Temp\ipykernel_43624\880300594.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 📝 自测清单

- [ ] 能计算相邻层转化率、整体转化率、流失率
- [ ] 能定位最大流失环节并给出排查方向
- [ ] 能把两个渠道漏斗叠对比，识别短板层
- [ ] 能解释「漏斗每一层的人数口径必须一致」